# EcoSort: Intelligent Waste Management Assistant — Summative Lab (NNs & Similar Models)

**Part 1:** Dataset exploration and preparation  
**Part 2:** Waste material classification with a CNN (transfer learning)  
**Part 3:** Waste description classification (text)  
**Part 4:** Recycling instruction generation with RAG  
**Part 5:** Integrated waste management assistant  

Authoring environment: Windows | Python 3.12 | CPU only (16 GB RAM)


In [ ]:
# %% [markdown]
# ## 0. Global configuration
# We centralise every path, class name, and hyper-parameter here so the whole
# notebook stays consistent and reproducible.

import os
import sys
import json
import re
import math
import random
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Quiet noisy TF startup logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import tensorflow as tf
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    else:
        print('INFO: No GPU detected - running on CPU.')
except Exception as e:
    print('GPU config skipped:', e)
tf.random.set_seed(42)

import torch
torch.manual_seed(42)

# ---------------------------------------------------------------- paths
BASE = os.getcwd()
DATA = os.path.join(BASE, 'data')          # derived data (splits, features)
FIG = os.path.join(DATA, 'figures')        # saved plots
MODELS = os.path.join(BASE, 'models')      # saved model artifacts
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

def find_realwaste_root():
    cands = [
        os.path.join(BASE, 'realwaste', 'realwaste-main', 'RealWaste'),
        os.path.join(BASE, 'RealWaste'),
        os.path.join(BASE, 'data', 'RealWaste'),
    ]
    for c in cands:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError('RealWaste image root not found. Please place the '
                            '"RealWaste" class folders next to the data.')
IMAGE_ROOT = find_realwaste_root()
TEXT_CSV = os.path.join(BASE, 'waste_descriptions.csv')
POLICY_JSON = os.path.join(BASE, 'waste_policy_documents.json')

CLASSES = sorted([d for d in os.listdir(IMAGE_ROOT)
                  if os.path.isdir(os.path.join(IMAGE_ROOT, d))])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(f'IMAGE_ROOT  : {IMAGE_ROOT}')
print(f'N classes   : {len(CLASSES)}')
print(f'Classes     : {CLASSES}')
print(f'TensorFlow  : {tf.__version__}   torch: {torch.__version__}')


In [ ]:
# %% [markdown]
# ### Shared helpers (used everywhere below)

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); torch.manual_seed(s)

def save_fig(fig, name, show=True, dpi=110, tight=True):
    """Save the figure and (for the notebook output) display it."""
    p = os.path.join(FIG, name)
    fig.savefig(p, dpi=dpi, bbox_inches='tight' if tight else None)
    if show:
        plt.show()
    plt.close(fig)
    return p

def clf_metrics(y_true, y_pred, labels):
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score, classification_report,
                                 confusion_matrix)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, digits=3),
        'confusion': confusion_matrix(y_true, y_pred, labels=range(len(labels))),
    }

def plot_confusion(cm, labels, title='Confusion matrix', cmap='Blues'):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels,
                yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    return fig


# Part 1: Dataset Exploration & Preparation

## 1.1 RealWaste image dataset

We start by building a tidy DataFrame over the **RealWaste** images
(9 material classes), scanning basic metadata (size, colour mode, file
size, brightness / saturation) so we can reason about the visual
characteristics and the challenges the CNN will face.


In [ ]:
# -------------------------------------------------
# 1.1.1 Build the image catalogue (cached on disk)
meta_path = os.path.join(DATA, 'image_metadata.csv')
if os.path.exists(meta_path):
    df_img = pd.read_csv(meta_path)
    print('Loaded cached image metadata.')
else:
    rows = []
    for cls in CLASSES:
        cdir = os.path.join(IMAGE_ROOT, cls)
        for fn in sorted(os.listdir(cdir)):
            p = os.path.join(cdir, fn)
            rows.append({'filename': fn, 'class': cls, 'path': p})
    df_img = pd.DataFrame(rows)

    def scan_row(r):
        try:
            im = Image.open(r['path'])
            w, h = im.size
            mode = im.mode
            fs = os.path.getsize(r['path'])
            small = im.convert('RGB').resize((48, 48))
            arr = np.asarray(small).astype(np.float32)
            hsv = np.asarray(small.convert('HSV')).astype(np.float32)
            im.close()
            return pd.Series({
                'width': w, 'height': h, 'mode': mode,
                'aspect_ratio': w / h, 'file_kb': round(fs / 1024, 1),
                'mean_brightness': arr.mean(),
                'std_brightness': arr.std(),
                'mean_saturation': hsv[..., 1].mean(),
            })
        except Exception as ex:
            return pd.Series({'width': np.nan, 'height': np.nan, 'mode': 'ERR',
                              'aspect_ratio': np.nan, 'file_kb': np.nan,
                              'mean_brightness': np.nan, 'std_brightness': np.nan,
                              'mean_saturation': np.nan})

    meta = df_img.apply(scan_row, axis=1)
    df_img = pd.concat([df_img, meta], axis=1)
    df_img.to_csv(meta_path, index=False)
    print('Catalogue written to', meta_path)

print(df_img.shape)
print('Classes:', df_img.groupby('class').size().to_dict())
print('NaNs in metadata:', df_img[['width', 'height', 'mean_brightness']].isna().sum().sum())


In [ ]:
# -------------------------------------------------
# 1.1.2 Class distribution
class_counts = df_img['class'].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(class_counts.index, class_counts.values, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('RealWaste - number of images per material class')
ax.set_ylabel('Images')
for b, v in zip(bars, class_counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 4, str(v), ha='center', fontsize=9)
plt.xticks(rotation=45, ha='right')
save_fig(fig, 'image_class_distribution.png')

imbalance = class_counts.max() / class_counts.min()
print(f'Most frequent class: {class_counts.index[0]} ({class_counts.iloc[0]})')
print(f'Least frequent class: {class_counts.index[-1]} ({class_counts.iloc[-1]})')
print(f'Class imbalance ratio (max/min) = {imbalance:.2f}x  -> class weights will be used.')

# -------------------------------------------------
# 1.1.3 Image geometry / quality summary
print('\n--- Image geometry  (all sizes are square) ---')
print(df_img[['width', 'height', 'aspect_ratio', 'file_kb']].describe().round(1).to_string())
print('\nColour modes:', df_img['mode'].value_counts().to_dict())

print('\n--- Per-class visual summaries (brightness 0-255, saturation 0-255) ---')
g = df_img.groupby('class')[['mean_brightness', 'mean_saturation', 'std_brightness', 'file_kb']].mean().round(1).sort_values('mean_brightness')
print(g.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_img, x='class', y='mean_brightness', ax=axes[0], palette='viridis')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Mean brightness per class')
sns.boxplot(data=df_img, x='class', y='mean_saturation', ax=axes[1], palette='magma')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title('Mean saturation per class')
save_fig(fig, 'image_brightness_saturation.png')


In [ ]:
# -------------------------------------------------
# 1.1.4 Look at real samples per class
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, cls in zip(axes.ravel(), CLASSES):
    sub = df_img[df_img['class'] == cls]
    sample = sub.iloc[random.randrange(len(sub))]['path']
    im = Image.open(sample).resize((180, 180))
    ax.imshow(im); ax.set_title(cls, fontsize=10); ax.axis('off')
plt.suptitle('Random sample from each RealWaste class', y=1.01)
save_fig(fig, 'image_class_samples.png')

# And one richer grid for the "Plastic vs Glass vs Metal" family which we
# expect to be confusable.
def grid_for(cls_subset, title, n=6):
    rng = random.Random(7)
    rows = []
    for cls in cls_subset:
        sub = df_img[df_img['class'] == cls]
        rows.append(sub.sample(n, random_state=7))
    panel = pd.concat(rows)
    fig, axes = plt.subplots(n, len(cls_subset), figsize=(3 * len(cls_subset), 3 * n))
    for i, cls in enumerate(cls_subset):
        for j, (_, r) in enumerate(panel[panel['class'] == cls].iterrows()):
            im = Image.open(r['path']).resize((120, 120))
            axes[j, i].imshow(im); axes[j, i].axis('off')
            if j == 0:
                axes[j, i].set_title(cls, fontsize=10)
    plt.suptitle(title, y=1.01)
    return fig

fig = grid_for(['Plastic', 'Glass', 'Metal', 'Miscellaneous Trash'],
               'Potentially confusable classes: Plastic vs Glass vs Metal vs Miscellaneous Trash')
save_fig(fig, 'image_confusable_grid.png')


## 1.2 Waste description text dataset (`waste_descriptions.csv`)

5,000 resident-style descriptions of waste items with:
`description`, `category`, `disposal_instruction`, `common_confusion`,
`material_composition`.


In [ ]:
# -------------------------------------------------
# 1.2.1 Load and inspect the description data
df_txt = pd.read_csv(TEXT_CSV)
print('Shape:', df_txt.shape)
print('\nDtypes:\n', df_txt.dtypes)
print('\nHead:')
print(df_txt.head(8).to_string())
print('\nMissing values:')
print(df_txt.isna().sum().to_string())

print('\nCategory distribution:')
ct = df_txt['category'].value_counts()
print(ct.to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
ct.sort_values().plot.barh(ax=ax, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('waste_descriptions.csv - category distribution')
ax.set_xlabel('Count')
plt.tight_layout()
save_fig(fig, 'text_class_distribution.png')

# -------------------------------------------------
# 1.2.2 Vocabulary & structure analysis
lengths = df_txt['description'].str.split().str.len()
print('\nDescription word-count: mean %.2f, std %.2f, min %d, max %d' %
      (lengths.mean(), lengths.std(), lengths.min(), lengths.max()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=range(0, 13), edgecolor='k', color='steelblue')
ax.set_title('Description length (words)')
ax.set_xlabel('words'); ax.set_ylabel('count')
save_fig(fig, 'text_description_length.png')

vocab = set()
df_txt['description'].str.lower().str.findall(r"[a-z']+").apply(lambda t: vocab.update(t))
print(f'Unique lowercase tokens across all descriptions: {len(vocab):,}')

# -------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer
STOP = set(__import__('sklearn.feature_extraction.text', fromlist=['ENGLISH_STOP_WORDS']).ENGLISH_STOP_WORDS)
top_words = {}
for cls in CLASSES:
    corpus = df_txt[df_txt['category'] == cls]['description'].tolist()
    cv = CountVectorizer(ngram_range=(1, 1), stop_words='english')
    try:
        X = cv.fit_transform(corpus)
    except ValueError:
        continue
    sums = np.asarray(X.sum(axis=0)).ravel()
    idx = np.argsort(sums)[::-1][:8]
    top_words[cls] = [(cv.get_feature_names_out()[i], int(sums[i])) for i in idx]

print('\nTop-8 characteristic tokens per class:')
for cls, toks in top_words.items():
    print(f'{cls:20s} -> ' + ', '.join(f'{w}({c})' for w, c in toks))


In [ ]:
# -------------------------------------------------
# 1.2.3 Auxiliary text fields (why the extra columns matter for RAG)
print('common_confusion - filled in', df_txt['common_confusion'].notna().sum(), 'of', len(df_txt), 'rows')
print('\nMost frequent confusion notes:')
cc = df_txt['common_confusion'].dropna()
print(Counter(cc).most_common(8))

print('\nUnique disposal instructions:', df_txt['disposal_instruction'].nunique())
print('Most frequent disposal instructions:')
dc = df_txt['disposal_instruction'].value_counts()
print(dc.head(8).to_string())

print('\nUnique material-composition strings:', df_txt['material_composition'].nunique())
mc = df_txt['material_composition'].value_counts()
print(mc.head(8).to_string())

print('\nSample rows for the ambiguous class "Miscellaneous Trash":')
print(df_txt[df_txt['category'] == 'Miscellaneous Trash'].sample(5, random_state=1)[
    ['description', 'disposal_instruction', 'common_confusion']].to_string())


## 1.3 Waste policy documents (`waste_policy_documents.json`)

Municipal regulations from *Metro City*. Each record has a `policy_type`,
`categories_covered`, `effective_date`, `jurisdiction` and the raw
`document_text` block. These documents become both the **retrieval corpus**
and the **supervision signal** for the RAG generator.


In [ ]:
# -------------------------------------------------
# 1.3.1 Load and organise the policy documents
with open(POLICY_JSON, 'r', encoding='utf-8') as f:
    policy_raw = json.load(f)
print('Number of policy documents:', len(policy_raw))
pf = pd.DataFrame(policy_raw)
print('\nColumns:', list(pf.columns))
print('\nHead of tabular fields:')
print(pf[['policy_id', 'policy_type', 'categories_covered', 'effective_date',
          'jurisdiction']].head(14).to_string())

pf['n_words'] = pf['document_text'].str.split().str.len()
pf['n_lines'] = pf['document_text'].str.count('\n') + 1
print('\nDocument length stats (words):')
print(pf['n_words'].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
pf['n_words'].plot.hist(bins=15, ax=axes[0], edgecolor='k', color='seagreen')
axes[0].set_title('Policy document length (words)')
pf['policy_type'].value_counts().sort_values()[:12].plot.barh(ax=axes[1], color='teal')
axes[1].set_title('Policy types')
plt.tight_layout()
save_fig(fig, 'policy_docs_summary.png')

cov = Counter()
for cl in pf['categories_covered']:
    for c in cl:
        cov[c] += 1
print('\nPolicies covering each category:')
print(pd.Series(cov).sort_values(ascending=False).to_string())

# -------------------------------------------------
# 1.3.2 Structure of a document (they use a recurring SKELETON)
print('\n--- Structure of a typical policy document ---')
print(policy_raw[3]['document_text'][:900])


### 1.4 Preliminary data-quality findings & challenges

**Images (RealWaste)**
1. **Uniform geometry** — every image is an RGB JPEG, exactly `524 x 524`.
   There is no resolution variation to handle, but real resident photos will
   differ; we therefore still use augmentation (flips, rotations, zoom) so the
   CNN does not overfit to the dataset's particular capture style.
2. **Class imbalance** — `Plastic` (921) is ~2.9x more frequent than
   `Textile Trash` (318). Vanilla training will bias toward frequent classes,
   so we balance the loss with *class weights*.
3. **Confusable material families** — transparent bottles mean *Plastic vs
   Glass* and *Paper vs Cardboard* look similar; *Miscellaneous Trash* is a
   catch-all with internally inconsistent visuals. Expect confusion along
   these pairs; we analyse this explicitly.
4. **Background clutter** — capture-style differences (napkins, hands,
   tables) act as domain noise; brightness/saturation vary strongly which is
   why colour augmentation matters.

**Text (`waste_descriptions.csv`)**
1. **Very short inputs** (mean ~5 words) — almost the whole signal lives in a
   handful of tokens (e.g. "bottle", "crushed", "grounds").
2. **Half of `common_confusion` is empty** (2,504 blanks) — we treat it as an
   optional pre-retrieval hint, never a required field.
3. **Adjective-led phrasing** ("soiled", "wrinkled", "intact") is irrelevant
   to material class and adds noise; we keep noun vocabulary but watch that
   TF-IDF does not latch onto these modifiers.
4. Regional/term variation (see `material_composition`) needs normalisation so
   the classifier and retriever agree.

**Policy documents (JSON)**
1. A small corpus (14 docs) but with a highly *regular skeleton*
   (ACCEPTABLE / NON-ACCEPTABLE / COLLECTION METHOD / PREPARATION /
   BENEFITS) which lets us chunk reliably.
2. Some documents cover **multiple categories** (e.g. Municipal Waste) and a
   few categories are *not* covered by their own dedicated document
   (e.g. Glass colours) — retrieval must therefore handle multi-category
   sections.
3. `categories_covered` gives us *ground-truth tags* to score retrieval.

These findings drive the preprocessing decisions made in the rest of Part 1.


### 1.5 Data pipelines for every modality

Now we create tidy, reproducible artifacts used by Parts 2-4:

* **Image split** — stratified 70 / 15 / 15 into `data/img_*.csv`, plus a
  `tf.data` pipeline (resize + augment + normalise).
* **Text split** — stratified 80 / 10 / 10 into `data/txt_*.csv` together with
  a light cleaning function used by both classical and neural models.
* **Policy chunking** — the 14 documents are split into per-category sections
  stored in `data/policy_chunks.csv`, ready to become RAG retrieval units.


In [ ]:
# ------------------------------------------------------
# 1.5.1 Image split (stratified) + tf.data pipeline
from sklearn.model_selection import train_test_split

train_df, tmp = train_test_split(df_img, test_size=0.30, stratify=df_img['class'],
                                 random_state=42)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['class'],
                                   random_state=42)
for name, d in [('img_train', train_df), ('img_val', val_df), ('img_test', test_df)]:
    d[['path', 'class', 'filename']].to_csv(os.path.join(DATA, name + '.csv'), index=False)
print('Image split sizes:', len(train_df), len(val_df), len(test_df))
print('Stratification check (class balances in each split):')
print(pd.DataFrame({
    'train': train_df['class'].value_counts(normalize=True),
    'val':   val_df['class'].value_counts(normalize=True),
    'test':  test_df['class'].value_counts(normalize=True),
}).round(3).to_string())

y_train_img = train_df['class'].map(CLASS_TO_IDX).values
class_weight_dict = {}
for cur in np.unique(y_train_img):
    n = (y_train_img == cur).sum()
    class_weight_dict[cur] = len(y_train_img) / (len(np.unique(y_train_img)) * n)
print('\nComputed image class weights (inverse frequency, normalised):')
print({IDX_TO_CLASS[k]: round(v, 3) for k, v in class_weight_dict.items()})

IMG_SIZE_DEFAULT = 224

def build_img_pipeline(df, img_size, batch_size, augment=False, cache_name=None):
    paths = df['path'].values
    labels = df['class'].map(CLASS_TO_IDX).values.astype(np.int64)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        raw = tf.io.read_file(p)
        x = tf.image.decode_jpeg(raw, channels=3)
        x = tf.image.resize(x, (img_size, img_size))
        return x, y
    if augment:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal'),
            tf.keras.layers.RandomRotation(0.15),
            tf.keras.layers.RandomZoom(0.15),
            tf.keras.layers.RandomBrightness(0.15),
            tf.keras.layers.RandomContrast(0.15),
        ], name='aug')
        def _a(x, y):
            x = tf.cast(x, tf.float32)
            return aug(x, training=True), y
    else:
        def _a(x, y):
            return tf.cast(x, tf.float32), y
    def _norm(x, y):
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        return tf.cast(x, tf.float32), y
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_a, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_norm, num_parallel_calls=tf.data.AUTOTUNE)
    if cache_name:
        ds = ds.cache(os.path.join(DATA, cache_name))
    ds = ds.shuffle(2048, reshuffle_each_iteration=True) if augment else ds
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds224 = build_img_pipeline(train_df, IMG_SIZE_DEFAULT, 32, augment=True, cache_name='cache_train_224')
val_ds224   = build_img_pipeline(val_df,   IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_val_224')
test_ds224  = build_img_pipeline(test_df,  IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_test_224')
print('Sample batch shapes (train):',
      [t.shape for t in next(iter(train_ds224.unbatch().batch(8)))])

# ------------------------------------------------------
# 1.5.2 Text cleaning + split
def clean_text(s):
    """Lightweight cleaning tuned for resident-style descriptions."""
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)      # urls
    s = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' ', s)   # emails
    s = re.sub(r'[^a-z\s]', ' ', s)                   # punctuation/numbers
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_txt['clean'] = df_txt['description'].apply(clean_text)
df_txt['n_words'] = df_txt['clean'].str.split().str.len()
print('Clean examples:')
print(df_txt[['description', 'clean']].head(5).to_string())

X_txt = df_txt['clean'].values.astype(str)
y_txt = df_txt['category'].astype('category').cat.codes.values
cat_names = list(df_txt['category'].astype('category').cat.categories)
for i, c in enumerate(cat_names):
    print(i, c)

Xtr, tmpX = train_test_split(np.arange(len(df_txt)), test_size=0.20, stratify=y_txt, random_state=42)
Xv, Xte = train_test_split(tmpX, test_size=0.50, stratify=y_txt[tmpX], random_state=42)
train_idx, val_idx, test_idx = Xtr, Xv, Xte
print('\nText split sizes:', len(train_idx), len(val_idx), len(test_idx))
for name, idx in [('txt_train', train_idx), ('txt_val', val_idx), ('txt_test', test_idx)]:
    pd.DataFrame({'index': idx, 'text': df_txt['clean'].iloc[idx].values,
                  'label': df_txt['category'].iloc[idx].values}).to_csv(
        os.path.join(DATA, name + '.csv'), index=False)

# ------------------------------------------------------
# 1.5.3 Policy documents -> per-category chunks (improved splitter)
def split_sections(doc_text):
    """Split a policy document on its recurring section headers."""
    lines = doc_text.split('\n')
    header_re = re.compile(r'^[A-Za-z][A-Za-z0-9 /\-]*:$')
    sections = []
    cur_head, cur = None, []
    for ln in lines:
        s = ln.strip()
        if s and len(s) <= 45 and header_re.match(s) and not s.startswith('-'):
            if cur_head is not None:
                sections.append((cur_head, '\n'.join(cur).strip()))
            cur_head, cur = s.rstrip(':').strip(), []
        else:
            cur.append(ln)
    if cur_head is not None:
        sections.append((cur_head, '\n'.join(cur).strip()))
    return sections

def assign_category(header, categories):
    h = header.lower()
    if len(categories) == 1:
        return categories[0]
    for c in categories:
        if c.lower() in h:
            return c
    return None  # e.g. GENERAL REQUIREMENTS section

CHUNKS = []
for doc in policy_raw:
    for header, body in split_sections(doc['document_text']):
        tag = assign_category(header, doc['categories_covered'])
        CHUNKS.append({
            'section': header, 'category': tag, 'text': body,
            'policy_id': doc['policy_id'], 'policy_type': doc['policy_type'],
            'categories_covered': ','.join(doc['categories_covered']),
            'effective_date': doc['effective_date'], 'jurisdiction': doc['jurisdiction'],
        })
chunk_df = pd.DataFrame(CHUNKS)
print('Policy chunks created:', chunk_df.shape)
print('\nTagged vs untagged chunks:')
print(chunk_df['category'].fillna('GENERAL').value_counts().to_string())
print('\nA few sample chunks:')
print(chunk_df[['section', 'category', 'text']].head(3).to_string())
print('\nChunk word-count stats:')
print(chunk_df['text'].str.split().str.len().describe().round(1).to_string())
chunk_df.to_csv(os.path.join(DATA, 'policy_chunks.csv'), index=False)


# Part 2: Waste Material Classification (CNN)

**Approach — transfer learning with MobileNetV2**

* **Why MobileNetV2:** state-of-the-art accuracy vs FLOPs ratio, built-in
  inverted-residual blocks, and it trains in reasonable time on a CPU-only
  machine (unlike ResNet-152 / EfficientNet-B4). Its 224px design matches the
  uniform 524px RealWaste captures well.
* **Two-stage strategy** (typical for small datasets):
  1. *Bottleneck feature extraction* at 224px with frozen ImageNet weights →
     fast, well-regularised head training, letting us experiment with head
     architectures cheaply;
  2. *Fine-tuning* the top convolutional block at low learning rate with
     input augmentation so the filters adapt to the waste domain.
* **Regularisation:** dropout in the head, data augmentation, L2 in the
  final layer is handled by Keras' default, early stopping, and a low
  fine-tune LR.
* **Class imbalance:** inverse-frequency class weights (Plastic is ~2.9x
  more frequent than Textile Trash).

> Compute note: we run on CPU, so fine-tuning uses 128px inputs (MobileNetV2
> accepts any size >= 32) to keep epochs tractable; the frozen 224px stage
> gives us the "native resolution" performance reference.


In [ ]:
# --------------------------------------------------------
# 2.1 Feature extraction (frozen ImageNet backbone), cached on disk
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

IMG_FEAT = 224
feat_path = os.path.join(DATA, 'img_features224.npz')
if os.path.exists(feat_path):
    z = np.load(feat_path)
    F_tr, F_va, F_te = z['F_tr'], z['F_va'], z['F_te']
    y_f_tr, y_f_va, y_f_te = z['y_tr'], z['y_va'], z['y_te']
    print('Loaded cached ImageNet-224 features:', F_tr.shape, F_va.shape, F_te.shape)
else:
    backbone = MobileNetV2(include_top=False, weights='imagenet',
                           input_shape=(IMG_FEAT, IMG_FEAT, 3), pooling='avg')
    backbone.trainable = False

    def feats_from(df_all, bs=64):
        paths = df_all['path'].values
        labels = df_all['class'].map(CLASS_TO_IDX).values.astype(np.int64)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        @tf.function
        def load(p, y):
            raw = tf.io.read_file(p)
            x = tf.image.decode_jpeg(raw, channels=3)
            x = tf.image.resize(x, (IMG_FEAT, IMG_FEAT))
            x = preprocess_input(tf.cast(x, tf.float32))
            return x, y
        ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(bs).prefetch(tf.data.AUTOTUNE)
        feats, ys = [], []
        steps = 0
        for xb, yb in ds:
            feats.append(backbone(xb, training=False).numpy())
            ys.append(yb.numpy())
            steps += 1
            if steps % 10 == 0:
                print(f'  extracted {steps*bs} samples...')
        return np.concatenate(feats), np.concatenate(ys)

    print('Extracting frozen features from all splits (224x224)...')
    t0 = time.time()
    F_tr, y_f_tr = feats_from(train_df)
    F_va, y_f_va = feats_from(val_df)
    F_te, y_f_te = feats_from(test_df)
    dt = time.time() - t0
    print(f'Feature extraction done in {dt/60:.1f} min '
          f'({F_tr.shape[1]}-dim vectors per image).')
    np.savez(feat_path, F_tr=F_tr, F_va=F_va, F_te=F_te,
             y_tr=y_f_tr, y_va=y_f_va, y_te=y_f_te)
    del backbone

print('Feature shapes -> train:', F_tr.shape, 'val:', F_va.shape, 'test:', F_te.shape)
print('Sample feature stats: mean %.4f  std %.4f' % (F_tr.mean(), F_tr.std()))


In [ ]:
# --------------------------------------------------------
# 2.2 Head-architecture experiments on the frozen features
# Evidence of hyper-parameter experimentation: width, depth & dropout.
from tensorflow.keras import layers, Model, Sequential as KSeq, Input as KInput
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

cw_feat = compute_class_weight('balanced', classes=np.unique(y_f_tr), y=y_f_tr)
cw_dict = {int(k): float(v) for k, v in zip(np.unique(y_f_tr), cw_feat)}

def make_head(dropout=0.3, hidden=128, bn=False):
    """Head mapping 1280-D bottleneck features -> 9 classes."""
    m = KSeq([KInput(shape=(F_tr.shape[1],))])
    m.add(layers.Dense(hidden, kernel_regularizer='l2'))
    if bn:
        m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.Dropout(dropout))
    m.add(layers.Dense(9, activation='softmax'))
    return m

variants = {
    'headA (128 units, drp 0.3)': dict(dropout=0.3, hidden=128, bn=False),
    'headB (256 units, drp 0.5, BN)': dict(dropout=0.5, hidden=256, bn=True),
    'headC (512 units, drp 0.6, BN)': dict(dropout=0.6, hidden=512, bn=True),
}

print('Training head variants on frozen 224px features (early-stopped):')
res_head = {}
_head_results = os.path.join(DATA, 'head_sweep_results.json')
if os.path.exists(_head_results):
    with open(_head_results) as f:
        res_head = json.load(f)
    print('Loaded cached head-sweep results:')
    print(pd.DataFrame(res_head).T.to_string())
    best_head = max(res_head, key=lambda k: res_head[k]['val_acc'])
    print('\nBest head variant (cached):', best_head)
else:
    for name, kw in variants.items():
        seed_everything(42)
        h = make_head(**kw)
        h.compile(Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
        h.fit(F_tr, y_f_tr, validation_data=(F_va, y_f_va), epochs=30, batch_size=64,
              class_weight=cw_dict, callbacks=[es], verbose=0)
        va = h.evaluate(F_va, y_f_va, verbose=0)[1]
        te = h.evaluate(F_te, y_f_te, verbose=0)[1]
        res_head[name] = {'val_acc': round(va, 4), 'test_acc': round(te, 4)}
        print(f'{name:28s} val_acc={va:.4f}  test_acc={te:.4f}')
        del h
    with open(_head_results, 'w') as fp:
        json.dump(res_head, fp, indent=2)

head_res = pd.DataFrame(res_head).T.sort_values('val_acc', ascending=False)
print(head_res.to_string())
best_head = head_res.index[0]
print('\nBest head variant:', best_head)


In [ ]:
# --------------------------------------------------------
# 2.3 Assemble the full transfer-learning model and fine-tune
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

# data pipelines at the fine-tune resolution (128 px) with augmentation
FT_SIZE = 128
train_ds = build_img_pipeline(train_df, FT_SIZE, 32, augment=True, cache_name='cache_train_128')
val_ds   = build_img_pipeline(val_df,   FT_SIZE, 32, augment=False)
test_ds  = build_img_pipeline(test_df,  FT_SIZE, 32, augment=False, cache_name='cache_test_128')

# class weights (same as used for the head)
cw_feat = compute_class_weight('balanced', classes=np.unique(y_f_tr), y=y_f_tr)
cw_dict = {int(k): float(v) for k, v in zip(np.unique(y_f_tr), cw_feat)}

FINAL_CNN_PATH = os.path.join(MODELS, 'cnn_finetuned.keras')
if os.path.exists(FINAL_CNN_PATH):
    print('Fine-tuned CNN checkpoint found - skipping the training stage '
          '(loads it in the next section).')
else:
    img_input = Input(shape=(FT_SIZE, FT_SIZE, 3))
    base = MobileNetV2(include_top=False, weights='imagenet',
                       input_shape=(FT_SIZE, FT_SIZE, 3), pooling='avg')
    base.trainable = False

    # best head architecture from the sweep (matched by name)
    def make_head_fixed(dropout=0.3, hidden=128, bn=False):
        m = KSeq([KInput(shape=(int(base.output.shape[1]),))])
        m.add(layers.Dense(hidden, kernel_regularizer='l2'))
        if bn:
            m.add(layers.BatchNormalization())
        m.add(layers.Activation('relu'))
        m.add(layers.Dropout(dropout))
        m.add(layers.Dense(9, activation='softmax'))
        return m

    choice = dict(hidden=128, dropout=0.3, bn=False)
    if best_head.startswith('headB'):
        choice = dict(hidden=256, dropout=0.5, bn=True)
    elif best_head.startswith('headC'):
        choice = dict(hidden=512, dropout=0.6, bn=True)
    print('Using head config:', choice, ' (winner of the sweep)')

    head = make_head_fixed(**choice)
    x = base(img_input)
    y = head(x)
    model = Model(img_input, y)
    model.compile(Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # 2.3.1 warm-up: train only the new head on augmented 128px images
    print('\n[Stage 1] Training the classification head (base frozen) ...')
    es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
    hist1 = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=cw_dict,
                      callbacks=[es], verbose=1)
    print(f'Head-only val_accuracy = {max(hist1.history["val_accuracy"]):.4f}')

    # 2.3.2 fine-tune the top convolutional block
    print('\n[Stage 2] Unfreezing the top 20 MobileNetV2 layers and fine-tuning ...')
    for layer in base.layers[-20:]:
        layer.trainable = True
    model.compile(Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cp = ModelCheckpoint(FINAL_CNN_PATH, monitor='val_accuracy', save_best_only=True)
    rlr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, min_lr=1e-7)
    hist2 = model.fit(train_ds, validation_data=val_ds, epochs=6, class_weight=cw_dict,
                      callbacks=[es, cp, rlr], verbose=1)
    print('Fine-tuned val_accuracy =', max(hist2.history['val_accuracy']))
    print('\nCNN model saved at:', FINAL_CNN_PATH)

final_cnn = tf.keras.models.load_model(FINAL_CNN_PATH)
print('Loaded final CNN for evaluation.')


In [ ]:
# --------------------------------------------------------
# 2.4 Evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# reload best checkpoint
final_cnn = tf.keras.models.load_model(os.path.join(MODELS, 'cnn_finetuned.keras'))
print('Loaded best fine-tuned CNN.')

pred_test = np.argmax(final_cnn.predict(test_ds, verbose=1), axis=1)
y_true = test_df['class'].map(CLASS_TO_IDX).values

acc = cnn_test_acc = accuracy_score(y_true, pred_test)
print(f'\nTest accuracy (fine-tuned CNN, 128px) = {cnn_test_acc:.4f}')

# head-only classifier evaluated on 224px frozen features (from the sweep)
acc_head = res_head[best_head]['test_acc']
print(f'Test accuracy (frozen features + {best_head}) = {acc_head:.4f}')

comp = pd.DataFrame({'stage': ['Frozen-224 + ' + best_head, 'Fine-tuned CNN (128px)'],
                     'test_accuracy': [round(acc_head, 4), round(cnn_test_acc, 4)]})
print('\n', comp.to_string(index=False))

# ---- confusion matrix + report for the final model ----
cm = confusion_matrix(y_true, pred_test, labels=range(len(CLASSES)))
fig = plot_confusion(cm, CLASSES, title='CNN confusion matrix - RealWaste test set')
save_fig(fig, 'cnn_confusion.png')

print('\nClassification report:')
print(classification_report(y_true, pred_test, target_names=CLASSES, digits=3, zero_division=0))

# per-class recall to spot the hard categories
per = pd.DataFrame({
    'class': CLASSES,
    'recall': [round((cm[i, i] / cm[i].sum()) if cm[i].sum() else 0, 3) for i in range(9)],
    'support': cm.sum(axis=1),
}).sort_values('recall')
print('\nHardest-to-easiest classes (by test recall):')
print(per.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
per.set_index('class')['recall'].plot.bar(ax=ax, color='firebrick')
ax.set_title('Per-class test recall (sorted)'); ax.set_ylabel('recall')
plt.xticks(rotation=45, ha='right')
save_fig(fig, 'cnn_per_class_recall.png')


In [ ]:
# --------------------------------------------------------
# 2.5 Error-pattern analysis: where does the CNN go wrong?
mis_idx = np.where(y_true != pred_test)[0]
print(f'Misclassified test images: {len(mis_idx)} of {len(y_true)} ({100*len(mis_idx)/len(y_true):.1f}%)')

wrong_pairs = pd.DataFrame({
    'true': [CLASSES[y_true[i]] for i in mis_idx],
    'pred': [CLASSES[pred_test[i]] for i in mis_idx],
})
pair_counts = wrong_pairs.value_counts().head(12)
print('\nMost common true->prediction error pairs:')
print(pair_counts.to_string())

# Show a handful of the actual failing images
rng = random.Random(3)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
picked = rng.sample(list(mis_idx), min(8, len(mis_idx)))
for ax, i in zip(axes.ravel(), picked):
    im = Image.open(test_df.iloc[i]['path']).resize((140, 140))
    ax.imshow(im)
    ax.set_title(f'TRUE: {CLASSES[y_true[i]]}\nPRED: {CLASSES[pred_test[i]]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Example CNN misclassifications on the held-out test set', y=1.0)
save_fig(fig, 'cnn_misclassification_examples.png')

# Which classes rob accuracy from others? (off-diagonal volumes)
off = cm.copy().astype(float)
for i in range(9):
    off[i, i] = 0
    if off[i].sum():
        off[i, :] = 100 * off[i, :] / off[i].sum()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(off, annot=True, fmt='.0f', cmap='Reds', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title('% of each TRUE class\'s errors landing on each PREDICTED class')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
save_fig(fig, 'cnn_error_distribution.png')
print('\nInterpretation: most confusion is concentrated along '
      'Plastic<->Miscellaneous Trash and Paper<->Cardboard.')


# Part 3: Waste Description Classification (text)

Descriptions are **very short** (mean ~5 words) and use a limited
vocabulary, so a dense TF-IDF representation is extremely effective.
We follow a two-track design:

1. **Classical ML track** (Option A in the brief) — Multinomial Naive Bayes,
   Logistic Regression and Random Forest over TF-IDF features, with a small
   hyper-parameter search (ngram range, regularisation) as evidence of
   experimentation.
2. **Neural track** — a compact learned-embedding *BiLSTM* classifier handled
   by Keras TextVectorization; it learns task-specific word embeddings
   directly.

The best system (by validation accuracy) is wrapped in a reusable
`classify_description(text)` function. A note: on a GPU, fine-tuning
**DistilBERT** on the same 5,000 examples is the natural transformer upgrade
to this pipeline, but on our CPU budget the BiLSTM + strong TF-IDF baselines
are the right engineering trade-off.


In [ ]:
# --------------------------------------------------------
# 3.1 Build TF-IDF features (fit ONLY on the training folds)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score

X_train_txt = df_txt['clean'].iloc[train_idx].values
y_train_txt = y_txt[train_idx]
X_val_txt   = df_txt['clean'].iloc[val_idx].values
y_val_txt   = y_txt[val_idx]
X_test_txt  = df_txt['clean'].iloc[test_idx].values
y_test_txt  = y_txt[test_idx]

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                        sublinear_tf=True, stop_words='english')
Xtr_t = tfidf.fit_transform(X_train_txt)
Xva_t = tfidf.transform(X_val_txt)
Xte_t = tfidf.transform(X_test_txt)
print('TF-IDF matrix (train):', Xtr_t.shape)
print('Vocab size:', len(tfidf.vocabulary_))

models = {
    'MultinomialNB': MultinomialNB(alpha=0.1),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=3000, n_jobs=-1),
    'RandomForest': RandomForestClassifier(n_estimators=250, n_jobs=-1, random_state=42),
}
val_accs = {}
for name, mdl in models.items():
    mdl.fit(Xtr_t, y_train_txt)
    va = accuracy_score(y_val_txt, mdl.predict(Xva_t))
    val_accs[name] = round(va, 4)
    print(f'{name:20s} val_acc = {va:.4f}')


In [ ]:
# --------------------------------------------------------
# 3.2 Hyper-parameter optimisation (evidence of experimentation)
param_grid = {
    'C': [0.1, 1.0, 10.0],
}
grid = GridSearchCV(LogisticRegression(max_iter=3000, n_jobs=-1), param_grid,
                    cv=StratifiedKFold(3, shuffle=True, random_state=42),
                    scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(Xtr_t, y_train_txt)
print('\nBest C for LogisticRegression:', grid.best_params_, 'cv_acc=%.4f' % grid.best_score_)

# Does character information add anything on top of word TF-IDF?
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)
Xtr_c = tfidf_char.fit_transform(X_train_txt)
Xva_c = tfidf_char.transform(X_val_txt)

from scipy.sparse import hstack
Xtr_h = hstack([Xtr_t, Xtr_c])
Xva_h = hstack([Xva_t, Xva_c])
for name, Xtr, Xva in [('word-level only', Xtr_t, Xva_t), ('word + char(3-5)', Xtr_h, Xva_h)]:
    lr = LogisticRegression(C=grid.best_params_['C'], max_iter=3000, n_jobs=-1)
    lr.fit(Xtr, y_train_txt)
    print(f'LogisticRegression on {name:20s} -> val_acc = {accuracy_score(y_val_txt, lr.predict(Xva)):.4f}')

# Winner so far: fine-tuned Logistic Regression on word TF-IDF
final_ml = LogisticRegression(C=grid.best_params_['C'], max_iter=3000, n_jobs=-1)
final_ml.fit(Xtr_t, y_train_txt)
print('\nChosen classical model trained on word TF-IDF.')


In [ ]:
# --------------------------------------------------------
# 3.3 Neural text classifier (embeddings + BiLSTM)
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping as ES2

from sklearn.utils.class_weight import compute_class_weight

SEQLEN = 22
vectorizer = TextVectorization(max_tokens=9000, output_mode='int',
                               output_sequence_length=SEQLEN)
vectorizer.adapt(X_train_txt.tolist())
print('Neural vocab size used:', vectorizer.vocabulary_size())

def build_bi_lstm(vocab_size, seqlen):
    m = Sequential([
        Embedding(vocab_size, 128, mask_zero=True, input_length=seqlen),
        Bidirectional(LSTM(64, dropout=0.3, return_sequences=False)),
        Dropout(0.3),
        Dense(9, activation='softmax'),
    ])
    return m

lstm_model = build_bi_lstm(vectorizer.vocabulary_size(), SEQLEN)
lstm_model.compile(Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

Xtr_n = vectorizer(X_train_txt.tolist())
Xva_n = vectorizer(X_val_txt.tolist())
Xte_n = vectorizer(X_test_txt.tolist())

cw_text = compute_class_weight('balanced', classes=np.unique(y_train_txt), y=y_train_txt)
cw_text_dict = {int(k): float(v) for k, v in zip(np.unique(y_train_txt), cw_text)}

lstm_model.fit(Xtr_n, y_train_txt, validation_data=(Xva_n, y_val_txt),
               epochs=14, batch_size=64, class_weight=cw_text_dict,
               callbacks=[ES2(monitor='val_accuracy', patience=3, restore_best_weights=True)],
               verbose=1)
val_nn = lstm_model.evaluate(Xva_n, y_val_txt, verbose=0)[1]
print(f'\nBiLSTM  val_acc = {val_nn:.4f}')

# save both winner artifacts
import joblib
joblib.dump({"model": final_ml, "tfidf": tfidf, "classes": list(CLASSES),
             "y_cat_names": list(df_txt['category'].astype('category').cat.categories)},
            os.path.join(MODELS, 'text_ml_pipeline.pkl'))
lstm_model.save(os.path.join(MODELS, 'text_bilstm.keras'))
print('Saved text model artifacts.')


In [ ]:
# --------------------------------------------------------
# 3.4 Evaluate the best text model on the held-out test set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
best_txt = final_ml
yhat = best_txt.predict(Xte_t)
acc = accuracy_score(y_test_txt, yhat)
print(f'Final text classifier (LogisticRegression + TF-IDF): test accuracy = {acc:.4f}')

yhat_nn = np.argmax(lstm_model.predict(Xte_n, verbose=0), axis=1)
nn_te = accuracy_score(y_test_txt, yhat_nn)
print(f'Text classifier (BiLSTM):                test accuracy = {nn_te:.4f}')

cat_names = list(df_txt['category'].astype('category').cat.categories)
cm = confusion_matrix(y_test_txt, yhat, labels=range(len(cat_names)))
fig = plot_confusion(cm, cat_names, title='Text classification confusion matrix (LogReg + TF-IDF)')
save_fig(fig, 'text_confusion.png')

print('\nClassification report (best text model):')
print(classification_report(y_test_txt, yhat, target_names=cat_names, digits=3, zero_division=0))


In [ ]:
# --------------------------------------------------------
# 3.5 Error-pattern analysis for text classification
from sklearn.metrics import accuracy_score as acc_fn

# The winning LogReg model is near-perfect on this synthetic-but-separable
# test set (see 3.4). To still expose patterns, we (a) study the errors of the
# *weaker* baselines, and (b) look at low-confidence corners of the winner.

# (a) errors of the weaker baselines -> where does text classification fail?
mnb = MultinomialNB(alpha=0.1).fit(Xtr_t, y_train_txt)
rf = RandomForestClassifier(n_estimators=250, n_jobs=-1, random_state=42).fit(Xtr_t, y_train_txt)

weak = pd.DataFrame({
    'y': y_test_txt,
    'mnb': mnb.predict(Xte_t),
    'rf': rf.predict(Xte_t),
    'lr': yhat,
})
weak['mnb_ok'] = weak['y'] == weak['mnb']
weak['rf_ok'] = weak['y'] == weak['rf']
weak['lr_ok'] = weak['y'] == weak['lr']
print('Test accuracy  NB=%.3f  RF=%.3f  LR=%.3f' % (
    weak['mnb_ok'].mean(), weak['rf_ok'].mean(), weak['lr_ok'].mean()))

def error_pairs(df, pred_col):
    m = df[df['y'] != df[pred_col]]
    return pd.DataFrame({'true': df.loc[m.index, 'y'].map(lambda c: cat_names[c]),
                         'pred': m[pred_col].map(lambda c: cat_names[c])})

pairs_mnb = error_pairs(weak, 'mnb')
pairs_rf = error_pairs(weak, 'rf')
for nm, pr in [('MultinomialNB', pairs_mnb), ('RandomForest', pairs_rf)]:
    print(f'\n{nm} - most common true->prediction error pairs:')
    print(pr.value_counts().head(8).to_string() if len(pr) else '  (no errors on the test fold)')

# examples of NB errors (most instructive: the "hard" descriptions)
print('\nExample descriptions the weaker baseline (NB) got wrong:')
if len(pairs_mnb):
    mis_mnb = weak.index[~weak['mnb_ok']].tolist()
    rng = random.Random(7)
    for i in rng.sample(mis_mnb, min(8, len(mis_mnb))):
        print(f'  [{cat_names[weak.loc[i,"y"]]:16s}->{cat_names[weak.loc[i,"mnb"]]:16s}]  '
              f'"{df_txt.iloc[test_idx[i]].description}"')

# (b) low-confidence corners of the WINNING model (still correct, but "hard")
probs = best_txt.predict_proba(Xte_t)
conf = probs.max(axis=1)
df_te = df_txt.iloc[test_idx].copy()
df_te['confidence'] = conf
df_te['correct'] = weak['lr_ok'].values

print('\nDistribution of winner model confidence on the test fold:')
print(df_te['confidence'].describe().round(3).to_string())
print('\n10 lowest-confidence test descriptions (challenging phrasing, '
      'even when classified correctly):')
for _, r in df_te.nsmallest(10, 'confidence').iterrows():
    print(f'  conf={r["confidence"]:.3f}  true={r["category"]:16s}  "{r["description"]}"')

# (c) does description length affect confidence?
bins = pd.cut(df_te['n_words'], [0, 2, 4, 6, 8, 20])
g = df_te.groupby(bins, observed=True)['confidence'].agg(['count', 'mean'])
fig, ax = plt.subplots(figsize=(7, 4))
g['mean'].plot.bar(ax=ax, color='darkorange')
ax.set_title('Average classifier confidence vs description length')
ax.set_ylabel('mean confidence'); ax.set_xlabel('n words')
save_fig(fig, 'text_conf_by_length.png')
print('\nMean confidence by description length bucket:')
print(g.round(3).to_string())

# store variables Part 5 reuses
pairs = pairs_lr = pd.DataFrame({
    'true': [cat_names[y_test_txt[i]] for i in np.where(y_test_txt != yhat)[0]],
    'pred': [cat_names[yhat[i]] for i in np.where(y_test_txt != yhat)[0]]})
pairs_worst = (', '.join(map(str, pairs.value_counts().index[0]))
               if len(pairs) else 'none (winner is perfect on test)')
print('\nWorst winner-model error pair:', pairs_worst, ' (uses weaker baselines for the '
      'error-pattern study, since the winner is at ~100% on this data)')


In [ ]:
# --------------------------------------------------------
# 3.6 Reusable classifier function (Part 5 integration point)
def classify_description(text: str):
    """Return (category, confidence, top3) for a free-text waste description."""
    cleaned = clean_text(text)
    if not cleaned:
        raise ValueError('Empty description after cleaning.')
    X = tfidf.transform([cleaned])
    probs = best_txt.predict_proba(X)[0]
    cls_order = best_txt.classes_           # actual class indices the LR saw
    order = np.argsort(probs)[::-1]
    top = [(cat_names[cls_order[i]], float(probs[i])) for i in order[:3]]
    return cat_names[cls_order[order[0]]], float(probs[order[0]]), top

for sample in ["a crushed aluminum cola can", "soggy cardboard takeaway box with grease",
               "empty clear glass bottle", "bunch of freshly cut grass"]:
    cat, conf, top3 = classify_description(sample)
    print(f'{sample!r} -> {cat} (conf={conf:.2f})  top3={[(c, round(p,2)) for c,p in top3]}')


# Part 4: Recycling Instruction Generation with RAG

We build a **Retrieval-Augmented Generation** pipeline:

1. **Retrieval corpus** — the Metro City policy documents (split into
   per-category sections) plus aggregated resident disposal advice, embedded
   with a sentence-transformer (`all-MiniLM-L6-v2`).
2. **Retrieval** — nearest-neighbour cosine search over the embedding index.
3. **Generation** — a pre-trained **DistilGPT2** LM fine-tuned (on ~200
   grounded examples built from the corpus) to produce recycling instructions
   that *rely on the retrieved reference text*.

The guiding design choice: because the generator is tiny, we deliberately
build its training objective as an **extractive grounding** task
(prompt-with-reference -> reference-derived answer). The model therefore has
to copy/summarise the retrieved policy rather than hallucinate instructions —
which is exactly what a small waste-management assistant should do.


In [ ]:
# --------------------------------------------------------
# 4.1 Build the retrieval corpus (policy sections + resident advice)
from sentence_transformers import SentenceTransformer

# policy sections (re-split robustly)
sections = []
for doc in policy_raw:
    for header, body in split_sections(doc['document_text']):
        if len(body.split()) < 5:
            continue
        tag = assign_category(header, doc['categories_covered'])
        sections.append({
            'doc_id': f"pol{doc['policy_id']}:{header}",
            'kind': 'policy',
            'category': tag,
            'valid_categories': [tag] if tag else doc['categories_covered'],
            'text': header + ':\n' + body,
            'policy_type': doc['policy_type'],
            'effective_date': doc['effective_date'],
        })

# aggregated, category-tagged "resident advice" documents from waste_descriptions.csv
# (disposal instructions + common confusions only; the synthetic
#  material-composition sentences are dropped - they are not actionable).
advice = []
for cat in CLASSES:
    sub = df_txt[df_txt['category'] == cat]
    instr = sub['disposal_instruction'].dropna().unique()
    instr = [i for i in instr if i and i.lower() not in ('recycle.', 'recycle', 'reuse.')][:12]
    conf = sub['common_confusion'].dropna().unique()[:5]
    txt = f'Resident disposal advice for {cat} waste items:\n- ' + '\n- '.join(instr)
    if len(conf):
        txt += '\nCommon confusions:\n- ' + '\n- '.join(conf)
    advice.append({'doc_id': f'advice:{cat}', 'kind': 'advice', 'category': cat,
                   'valid_categories': [cat], 'text': txt,
                   'policy_type': 'residential-advice', 'effective_date': ''})

corpus = pd.DataFrame(sections + advice)
print('Retrieval corpus:', corpus.shape)
print('\nKind balance:\n', corpus['kind'].value_counts().to_string())
print('\nDocs per category (policy sections tagged):')
print(corpus[corpus['kind'] == 'policy']['category'].fillna('GENERAL').value_counts().to_string())

# ---- category-tagged *index text* improves retrieval specificity.
# The generic section headers ("Collection Method") do not mention the
# material, so pure section embeddings can collapse across categories.
# Pre-pending the category gives the embedding key discriminative identity.
def _prefix(row):
    if row['category']:
        return row['category']
    return row['valid_categories'][0] if row['valid_categories'] else 'general'

corpus['index_text'] = [
    f"{_prefix(r)} recycling guidelines.\n{r['text']}" for _, r in corpus.iterrows()]

# ---- embed the index text
model_name = 'all-MiniLM-L6-v2'
encoder = SentenceTransformer(model_name)
emb = encoder.encode(corpus['index_text'].tolist(), normalize_embeddings=True,
                     show_progress_bar=True, batch_size=32)
np.savez(os.path.join(DATA, 'rag_embeddings.npz'), emb=emb)
corpus.drop(columns=['index_text']).to_csv(os.path.join(DATA, 'rag_corpus.csv'), index=False)
print('\nEmbedding matrix:', emb.shape, ' (L2-normalised)')
print('Sample corpus doc:\n', corpus.iloc[0]['text'][:400])


In [ ]:
# --------------------------------------------------------
# 4.2 Retrieval index + query functions
emb = np.load(os.path.join(DATA, 'rag_embeddings.npz'))['emb']
corpus = pd.read_csv(os.path.join(DATA, 'rag_corpus.csv'))

def retrieve(query, k=3):
    """Pure neural (embedding) retrieval."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sim = emb @ q                      # cosine because both are L2-normalised
    top = np.argsort(sim)[::-1][:k]
    return [(sim[i], int(i)) for i in top]

def retrieve_hybrid(query, category, k=3, boost=0.6):
    """Neural retrieval + category-aware boost.

    The upstream classifier already tells us the material class, so we
    preferentially retrieve the policy units tagged with that class. This
    metadata-aware hybrid consistently beats the embedded-only ranking for
    odd categories such as 'Miscellaneous Trash' (see evaluation below).
    """
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sim = emb @ q
    for i, r in corpus.iterrows():
        if valid_for(r, category):
            sim[i] *= (1 + boost)
    top = np.argsort(sim)[::-1][:k]
    return [(sim[i], int(i)) for i in top]

def valid_for(doc_row, category):
    vc = doc_row['valid_categories'] if isinstance(doc_row['valid_categories'], list) else []
    if isinstance(doc_row['valid_categories'], str):
        vc = [x.strip() for x in doc_row['valid_categories'].strip('[]').replace("'", "").split(',')]
    return category in vc

# ---- retrieval quality: hit / precision@k for each of the 9 categories
def _ev(x, k=3):
    return {'hit@k': len(x) > 0, 'p@k': round(len(x) / k, 3)}

pure_rows, hybrid_rows = [], []
for cat in CLASSES:
    for qv in [f'How to recycle {cat}?', f'what bin should I use for this {cat} item?']:
        pure = [i for _, i in retrieve(qv, k=3)]
        hyb = [i for _, i in retrieve_hybrid(qv, cat, k=3)]
        rel_p = [i for i in pure if valid_for(corpus.iloc[i], cat)]
        rel_h = [i for i in hyb if valid_for(corpus.iloc[i], cat)]
        pure_rows.append({'query': qv, 'category': cat, **_ev(rel_p, 3),
                          'best_doc': corpus.iloc[pure[0]]['doc_id']})
        hybrid_rows.append({'query': qv, 'category': cat, **_ev(rel_h, 3),
                            'best_doc': corpus.iloc[hyb[0]]['doc_id']})

ret_ev = pd.DataFrame(pure_rows)
ret_ev_hyb = pd.DataFrame(hybrid_rows)
print('--- Pure embedding retrieval ---')
print(ret_ev[['category', 'hit@k', 'p@k', 'best_doc']].to_string(index=False))
print(f'\nPURE retrieval   -> hit@3 = {ret_ev["hit@k"].mean():.2f}  '
      f'precision@3 = {ret_ev["p@k"].mean():.3f}')
print('\n--- Category-aware hybrid retrieval (used by the assistant) ---')
print(ret_ev_hyb[['category', 'hit@k', 'p@k', 'best_doc']].to_string(index=False))
print(f'\nHYBRID retrieval -> hit@3 = {ret_ev_hyb["hit@k"].mean():.2f}  '
      f'precision@3 = {ret_ev_hyb["p@k"].mean():.3f}')

# human-readable example of retrieval for one category
qv, cat = 'How to recycle Miscellaneous Trash?', 'Miscellaneous Trash'
print(f'\nExample query: {qv!r}  (category-aware hybrid)')
for s, i in retrieve_hybrid(qv, cat, k=3):
    r = corpus.iloc[i]
    print(f'  [sim {s:.3f}] {r["doc_id"]} | {r["text"][:100].replace(chr(10), " ")}')


In [ ]:
# --------------------------------------------------------
# 4.3 Build a grounded instruction corpus for fine-tuning the generator
# Examples: (question about a category) + (reference text) -> answer extracted
# from that same reference, so the model learns to ground its reply.
TEMPLATES = [
    "You are a recycling assistant for Metro City. Answer using only the reference policy.\nQuestion: How to recycle {cat}?\nReference:\n{ctx}\nAnswer:",
    "Recycling query from a resident: \"Can I recycle {cat} and how should I prepare it?\"\nPolicy reference:\n{ctx}\nRecycling instructions:",
    "Metro City policy lookup.\nQuestion: How to recycle {cat}?\nReference policy:\n{ctx}\nAnswer:",
]

def build_gen_examples():
    ex = []
    for _, r in corpus.iterrows():
        cats = r['valid_categories'] if isinstance(r['valid_categories'], list) else \
               [x.strip() for x in r['valid_categories'].strip('[]').replace("'", "").split(',')] \
               if isinstance(r['valid_categories'], str) else []
        for c in cats:
            if not c:
                continue
            ctx = r['text']
            for tpl in TEMPLATES:
                prompt = tpl.format(cat=c, ctx=ctx)
                ex.append({'prompt': prompt, 'target': ctx,
                           'category': c, 'doc_id': r['doc_id']})
    return pd.DataFrame(ex)

gen_df = build_gen_examples()
print('Instruction examples:', gen_df.shape)
print(gen_df[['prompt', 'target']].head(2).to_string())
# a 90/10 train/val split
gi = np.arange(len(gen_df))
tg = gen_df['category'].values
gtr, gtmp = train_test_split(gi, test_size=0.10, stratify=tg, random_state=42)
gval, _ = train_test_split(gtmp, test_size=0.5, stratify=tg[gtmp], random_state=42)
print('Fine-tune split sizes: train', len(gtr), 'val', len(gval))
gen_df.iloc[gtr].to_csv(os.path.join(DATA, 'rag_gen_train.csv'), index=False)
gen_df.iloc[gval].to_csv(os.path.join(DATA, 'rag_gen_val.csv'), index=False)


In [ ]:
# --------------------------------------------------------
# 4.4 Fine-tune DistilGPT2 on the grounded instruction corpus (CPU)
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import __version__ as hf_version

MODEL_NAME = 'distilgpt2'
GCKPT = os.path.join(MODELS, 'rag_gpt2_finetuned')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
tok.padding_side = 'right'
print('Tokenizer ready, HF transformers', hf_version)

gen_train = pd.read_csv(os.path.join(DATA, 'rag_gen_train.csv'))
gen_val = pd.read_csv(os.path.join(DATA, 'rag_gen_val.csv'))
print('Train/Val examples to fine-tune:', len(gen_train), len(gen_val))

# Idempotent cell: if a previously fine-tuned checkpoint exists we re-use it.
cp_ready = os.path.exists(os.path.join(GCKPT, 'config.json')) and (
    os.path.exists(os.path.join(GCKPT, 'model.safetensors')) or
    os.path.exists(os.path.join(GCKPT, 'pytorch_model.bin')))
if cp_ready:
    print('Fine-tuned generator checkpoint found - skipping the slow training '
          'step (loaded below).')
else:
    MAX_LEN = 384

    def tokenize_for_lm(df):
        ids = []
        for prompt, target in zip(df['prompt'], df['target']):
            text = prompt + ' ' + target
            full = tok.encode(text, truncation=True, max_length=MAX_LEN)
            ids.append(full)
        return ids

    train_ids = tokenize_for_lm(gen_train)
    val_ids = tokenize_for_lm(gen_val)

    class SeqDS(Dataset):
        def __init__(self, sequences):
            self.sequences = sequences
        def __len__(self):
            return len(self.sequences)
        def __getitem__(self, i):
            return self.sequences[i]

    def collate_fn(batch):
        # label every pad token with -100 so the LM loss ignores padding
        maxn = max(len(b) for b in batch)
        inp = np.full((len(batch), maxn), tok.eos_token_id, dtype=np.int64)
        lab = np.full((len(batch), maxn), -100, dtype=np.int64)
        for i, b in enumerate(batch):
            inp[i, :len(b)] = b
            lab[i, :len(b)] = b
        return torch.tensor(inp), torch.tensor(lab)

    train_dl = DataLoader(SeqDS(train_ids), batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_dl = DataLoader(SeqDS(val_ids), batch_size=8, shuffle=False, collate_fn=collate_fn)

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    def evaluate_loss(model, dl):
        model.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for inp, lab in dl:
                out = model(input_ids=inp, labels=lab)
                tot += out.loss.item() * inp.size(0)
                n += inp.size(0)
        model.train()
        return tot / n

    EPOCHS = 3
    best_val = 1e9
    print('\nFine-tuning on CPU (this is the slowest RAG step)...')
    for ep in range(1, EPOCHS + 1):
        t0ep = time.time()
        tr_loss, nb = 0.0, 0
        for step, (inp, lab) in enumerate(train_dl):
            out = model(input_ids=inp, labels=lab)
            loss = out.loss / 2          # gradient accumulation x2
            loss.backward()
            if (step + 1) % 2 == 0:
                opt.step(); opt.zero_grad()
            tr_loss += out.loss.item() * inp.size(0)
            nb += inp.size(0)
        opt.step(); opt.zero_grad()
        vloss = evaluate_loss(model, val_dl)
        print(f'epoch {ep} | train_loss={tr_loss/nb:.4f} val_loss={vloss:.4f} | {time.time()-t0ep:.0f}s')
        if vloss < best_val:
            best_val = vloss
            model.save_pretrained(GCKPT)
            tok.save_pretrained(GCKPT)
    model.save_pretrained(GCKPT); tok.save_pretrained(GCKPT)
    print('Best generator checkpoint saved to', GCKPT)


In [ ]:
# --------------------------------------------------------
# 4.5 Generate with different sampling settings + pick the best config
gen_model = AutoModelForCausalLM.from_pretrained(GCKPT)
gen_model.eval()

def make_prompt(category, k_docs=3):
    ctx = []
    # hybrid retrieval: the category is known, so retrieve its policy units first
    for s, i in retrieve_hybrid(f'How to recycle {category}?', category, k=k_docs):
        ctx.append(corpus.iloc[i]['text'])
    joined = '\n\n'.join(ctx)
    return (f'You are a recycling assistant for Metro City. Answer using only the reference policy.\n'
            f'Question: How to recycle {category}?\nReference:\n{joined}\nAnswer:'), joined

def generate(category, config, k_docs=3, max_new=110):
    prompt, ctx = make_prompt(category, k_docs)
    ids = tok.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        out = gen_model.generate(
            ids, max_new_tokens=max_new,
            do_sample=config.get('do_sample', False),
            temperature=config.get('temperature', 1.0),
            top_p=config.get('top_p', 1.0),
            top_k=config.get('top_k', 0),
            no_repeat_ngram_size=3, repetition_penalty=1.1,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id,
        )
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    cut = text.find('Question:')
    if cut > 0:
        text = text[:cut].strip()
    return text, ctx

def _toks(t):
    return re.findall(r"[a-z0-9']+", t.lower())

def jaccard(a, b):
    A, B = set(_toks(a)), set(_toks(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

configs = {
    'greedy':        {'do_sample': False},
    'temp=0.3 top_p=0.85': {'do_sample': True, 'temperature': 0.3, 'top_p': 0.85, 'top_k': 0},
    'temp=0.7 top_p=0.90': {'do_sample': True, 'temperature': 0.7, 'top_p': 0.90, 'top_k': 0},
    'temp=1.0 top_k=40':   {'do_sample': True, 'temperature': 1.0, 'top_p': 1.0, 'top_k': 40},
}

cmp_rows = []
samples_store = {}
for cname, cfg in configs.items():
    jac = 0.0; ln = 0
    for cat in CLASSES:
        text, ctx = generate(cat, cfg, k_docs=3)
        text = text[:300]
        jac += jaccard(text, ctx)
        ln += len(text.split())
    jac /= len(CLASSES); ln /= len(CLASSES)
    cmp_rows.append({'config': cname, 'faithfulness_Jaccard@ctx': round(jac, 3),
                     'avg_generated_words': round(ln, 1)})
    samples_store[cname] = generate('Plastic', cfg)[0][:200]

cmp_df = pd.DataFrame(cmp_rows).sort_values('faithfulness_Jaccard@ctx', ascending=False)
print('Sampling-configuration comparison (higher Jaccard = more grounded in retrieved policy):')
print(cmp_df.to_string(index=False))
best_cfg = cmp_df.iloc[0]['config']
print('\nBest config:', best_cfg)
GEN_CFG = configs[best_cfg]

print('\nSame query, different configs (Plastic):\n')
for cname, cfg in configs.items():
    print(f'--- {cname} ---\n{samples_store[cname]}\n')


In [ ]:
# --------------------------------------------------------
# 4.6 End-to-end RAG evaluation on all 9 categories
def rouge_n(ref, hyp, n):
    rg = Counter(zip(*[_toks(ref)[i:] for i in range(n)]))
    hg = Counter(zip(*[(_toks(hyp) + [''] * (n - 1))[i:] for i in range(n)]))
    if not rg or not hg:
        return 0.0
    ov = sum(min(rg[g], hg[g]) for g in hg)
    return ov / sum(rg.values()) if sum(rg.values()) else 0.0

def lcs_words(ref, hyp):
    a, b = _toks(ref), _toks(hyp)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i+1][j+1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j+1], dp[i+1][j])
    return dp[len(a)][len(b)]

ev = []
for cat in CLASSES:
    text, ctx = generate(cat, GEN_CFG, k_docs=3)
    top1 = retrieve_hybrid(f'How to recycle {cat}?', cat, k=1)[0][1]
    ref = corpus.iloc[top1]['text']
    ev.append({
        'category': cat,
        'generated_words': len(text.split()),
        'R1_vs_top1': round(rouge_n(ref, text, 1), 3),
        'R2_vs_top1': round(rouge_n(ref, text, 2), 3),
        'ROUGE_L_vs_top1': round(lcs_words(ref, text) / max(1, len(_toks(ref))), 3),
        'jaccard_vs_ctx': round(jaccard(text, ctx), 3),
        'generated': text,
        'retrieved': ref[:160],
    })
ragev = pd.DataFrame(ev)
print(ragev[['category', 'generated_words', 'R1_vs_top1', 'R2_vs_top1',
             'ROUGE_L_vs_top1', 'jaccard_vs_ctx']].to_string(index=False))
print('\nAverages:')
print(ragev[['R1_vs_top1', 'R2_vs_top1', 'ROUGE_L_vs_top1', 'jaccard_vs_ctx']].mean().round(3).to_string())

ragev.to_csv(os.path.join(DATA, 'rag_evaluation_results.csv'), index=False)

# human-readability: output three full answers for review
print('\n========= Sample generated instructions =========\n')
for cat in ['Paper', 'Glass', 'Miscellaneous Trash']:
    row = ragev[ragev['category'] == cat].iloc[0]
    print(f'### {cat}\nQuestion: How to recycle {cat}?\n', row['generated'], '\n')


### RAG summary

* **Retrieval** — at `hit@3` the correct material-specific guidance is almost
  always retrieved (see table above); the retrieval embedding generalises the
  "How to recycle X" template to paraphrased resident phrasings.
* **Generation** — the fine-tuned DistilGPT2 (a tiny CPU-feasible causal LM)
  produces short instruction paragraphs whose *vocabulary closely overlaps the
  retrieved policy text* (`ROUGE-L`/`Jaccard` statistics above). Because the
  fine-tuning set was deliberately *extractive*, the model effectively
  re-emits/condenses the retrieved Metro City policy instead of inventing
  facts — the core property we wanted from a small RAG assistant.
* **Limitations** — the generator can still drift into generic phrasing,
  hyphenate oddly, or truncate mid-instruction; a larger LM (e.g. DistilGPT2
  at higher epochs, or any instruct-tuned Llama) would read more fluently.
  The retrieval + reference-document *grounding* is what keeps answers
  factually aligned with Metro City policy (we return the sources below).


In [ ]:
# --------------------------------------------------------
# 4.7 Exposed function: given a waste category, produce instructions + sources
def generate_recycling_instructions(category, query=None, k_docs=3):
    """Return (instructions, sources) for a waste category using RAG."""
    query = query or f'How to recycle {category}?'
    text, ctx = generate(category, GEN_CFG, k_docs=k_docs)
    sources = []
    for s, i in retrieve_hybrid(query, category, k=k_docs):
        r = corpus.iloc[i]
        sources.append({'policy_type': r['policy_type'], 'doc_id': r['doc_id'],
                        'score': round(float(s), 3), 'excerpt': r['text'][:180]})
    return text, sources

for cat in ['Metal', 'Vegetation']:
    instr, src = generate_recycling_instructions(cat)
    print(f'=== {cat} ===\n{instr}\nSources: {[s["doc_id"] for s in src]}\n')


# Part 5: Integrated Waste Management Assistant

The three trained components are composed into a single assistant that
accepts **either an image or a free-text description** and returns
**(i)** the predicted waste category, **(ii)** tailored recycling
instructions and **(iii)** the supporting policy documents.

```text
                    +-------------------------------------------+
                    |         EcoSort Waste Assistant            |
                    |                                           |
  image  ---------> |  CNN classifier (MobileNetV2 + head)      |
  (uploaded photo)  |      -> predicted material class          |
                    |                                           |
  text   ---------> |  Text classifier (TF-IDF + LogisticReg)    |
  (resident query)  |      -> predicted material class          |
                    |                    |                      |
                    |                    v                      |
                    |            unified category               |
                    |                    |                      |
                    |                    v                      |
                    |  RAG generator: retrieve relevant         |
                    |  Metro City policy -> grounded recycling  |
                    |  instructions + source documents          |
                    +-------------------------------------------+
```

We also test robustness with **edge cases** (missing file, empty text,
low-confidence predictions) and sketch a lightweight **user-feedback**
mechanism that adapts retrieval scoring over time.


In [ ]:
# --------------------------------------------------------
# 5.1 Assemble the assistant (loads all saved artifacts)
import joblib
from sklearn.metrics import accuracy_score as _acc

class EcoSortAssistant:
    """Unified waste assistant: image OR text in -> instructions out."""

    MODELS = ['cnn', 'text_ml', 'text_nn', 'rag']

    def __init__(self, models_dir=None):
        if models_dir is None:
            models_dir = MODELS
        self.models_dir = models_dir
        self.labels = CLASSES
        self.idx_to_label = IDX_TO_CLASS

        # --- CNN ---
        self.cnn = tf.keras.models.load_model(os.path.join(models_dir, 'cnn_finetuned.keras'))
        self.cnn_size = FT_SIZE  # fine-tune resolution used at training time

        # --- text classifier (pick the best saved pipeline) ---
        pkl = joblib.load(os.path.join(models_dir, 'text_ml_pipeline.pkl'))
        self.tfidf = pkl['tfidf']
        self.text_clf = pkl['model']
        self.cat_names = list(pkl['y_cat_names'])
        self.bilstm = tf.keras.models.load_model(os.path.join(models_dir, 'text_bilstm.keras'))
        self.vec_nn = None  # set on first neural call (TextVectorization re-adapt)

        # --- RAG ---
        self.emb = np.load(os.path.join(DATA, 'rag_embeddings.npz'))['emb']
        self.corpus = pd.read_csv(os.path.join(DATA, 'rag_corpus.csv'))
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tok = AutoTokenizer.from_pretrained(os.path.join(models_dir, 'rag_gpt2_finetuned'))
        self.tok.pad_token = self.tok.eos_token
        self.gen_model = AutoModelForCausalLM.from_pretrained(os.path.join(models_dir, 'rag_gpt2_finetuned'))
        self.gen_model.eval()

        self.feedback_log = os.path.join(DATA, 'feedback_log.jsonl')
        self.confidence_threshold = 0.60

    # ---------------- image classification ----------------
    def predict_image(self, image_path):
        if not os.path.exists(image_path):
            raise FileNotFoundError(f'Image not found: {image_path}')
        im = Image.open(image_path).convert('RGB').resize((self.cnn_size, self.cnn_size))
        x = np.asarray(im, dtype=np.float32)
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        probs = self.cnn.predict(x[None], verbose=0)[0]
        order = np.argsort(probs)[::-1]
        return self.labels[order[0]], float(probs[order[0]]), \
               [(self.labels[i], float(probs[i])) for i in order[:3]]

    # ---------------- text classification ----------------
    def predict_text(self, text):
        cleaned = clean_text(text)
        if not cleaned:
            raise ValueError('Empty description after cleaning.')
        X = self.tfidf.transform([cleaned])
        probs = self.text_clf.predict_proba(X)[0]
        cls_order = self.text_clf.classes_
        order = np.argsort(probs)[::-1]
        return self.cat_names[cls_order[order[0]]], float(probs[order[0]]), \
               [(self.cat_names[cls_order[i]], float(probs[i])) for i in order[:3]]

    # ---------------- RAG ----------------
    def _retrieve(self, query, k=3, category=None, boost=0.6):
        q = self.encoder.encode([query], normalize_embeddings=True)[0]
        sim = self.emb @ q
        if category is not None:
            for i, r in self.corpus.iterrows():
                vc = str(r['valid_categories'])
                if category in [x.strip() for x in vc.strip('[]').replace("'", "").split(',')]:
                    sim[i] *= (1 + boost)
        top = np.argsort(sim)[::-1][:k]
        return [(float(sim[i]), int(i)) for i in top]

    def _generate(self, category, query, k_docs=3, max_new=110):
        ctx_parts = []
        for s, i in self._retrieve(query, k_docs, category=category):
            ctx_parts.append(self.corpus.iloc[i]['text'])
        joined = '\n\n'.join(ctx_parts)
        prompt = (f'You are a recycling assistant for Metro City. Answer using only the reference policy.\n'
                  f'Question: How to recycle {category}?\nReference:\n{joined}\nAnswer:')
        ids = self.tok.encode(prompt, return_tensors='pt')
        with torch.no_grad():
            out = self.gen_model.generate(ids, max_new_tokens=max_new,
                                          **GEN_CFG, no_repeat_ngram_size=3,
                                          repetition_penalty=1.1,
                                          eos_token_id=self.tok.eos_token_id,
                                          pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
        cut = text.find('Question:')
        if cut > 0:
            text = text[:cut].strip()
        return text, joined

    def instructions(self, category, query=None, k_docs=3):
        query = query or f'How to recycle {category}?'
        text, _ = self._generate(category, query, k_docs=k_docs)
        sources = [{'doc_id': self.corpus.iloc[i]['doc_id'],
                    'policy_type': self.corpus.iloc[i]['policy_type'],
                    'score': round(s, 3),
                    'excerpt': self.corpus.iloc[i]['text'][:160]}
                   for s, i in self._retrieve(query, k_docs, category=category)]
        return text, sources

    # ---------------- unified entry point ----------------
    def assist(self, image_path=None, text=None, k_docs=3):
        """Classify the input (image XOR text) and produce instructions."""
        history = []
        if image_path is not None:
            cat, conf, top3 = self.predict_image(image_path)
            history.append({'mode': 'image', 'input': os.path.basename(image_path),
                            'top3': top3})
        elif text is not None:
            cat, conf, top3 = self.predict_text(text)
            history.append({'mode': 'text', 'input': text, 'top3': top3})
        else:
            raise ValueError('Provide either image_path or text.')

        warning = None
        if conf < self.confidence_threshold:
            warning = (f'Low confidence ({conf:.2f}); the answer below uses the '
                       f'predicted category but treat it as provisional.')

        instr, sources = self.instructions(cat, k_docs=k_docs)
        return {'category': cat, 'confidence': conf, 'history': history,
                'warning': warning, 'instructions': instr, 'sources': sources}

    # ---------------- user feedback mechanism ----------------
    def log_feedback(self, category, note='', ok=True):
        with open(self.feedback_log, 'a', encoding='utf-8') as f:
            f.write(json.dumps({'ts': time.time(), 'category': category,
                                'satisfied': bool(ok), 'note': note}) + '\n')

    def apply_feedback_to_retrieval(self):
        """Reward categories that residents mark as useful when retrieving."""
        if not os.path.exists(self.feedback_log):
            return None
        scores = Counter()
        with open(self.feedback_log, encoding='utf-8') as f:
            for ln in f:
                r = json.loads(ln)
                if r.get('satisfied'):
                    scores[r['category']] += 1
                    scores[r['category']] += 0
        boosts = {c: 1.0 + 0.05 * min(s, 10) for c, s in scores.items()}
        return boosts

assistant = EcoSortAssistant()
print('EcoSortAssistant initialised.')
print('Quick smoke test (text):',
      assistant.assist(text='an empty crushed plastic bottle')['category'])


In [ ]:
# --------------------------------------------------------
# 5.2 System test on REAL test-set images
import matplotlib as _mpl
rng = random.Random(5)
demo = test_df.sample(3, random_state=5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (_, row) in zip(axes, demo.iterrows()):
    res = assistant.assist(image_path=row['path'])
    im = Image.open(row['path']).resize((200, 200))
    ax.imshow(im); ax.axis('off')
    ax.set_title(f"TRUE:{row['class']}  PRED:{res['category']}", fontsize=9)
plt.suptitle('Integrated assistant on held-out test images', y=1.0)
save_fig(fig, 'assistant_image_demo.png')

for _, row in demo.iterrows():
    print('\n' + '=' * 80)
    print(f"INPUT IMAGE  : {os.path.basename(row['path'])}  (TRUE: {row['class']})")
    res = assistant.assist(image_path=row['path'])
    print(f"PREDICTED    : {res['category']}  (conf {res['confidence']:.2f})")
    if res['warning']:
        print('WARNING      :', res['warning'])
    print('INSTRUCTIONS :')
    print('   ' + res['instructions'].replace('\n', '\n   '))
    print('SOURCES      :', [s['doc_id'] for s in res['sources']])


In [ ]:
# --------------------------------------------------------
# 5.3 System test on real text descriptions (from the held-out fold)
demo_txt = df_txt.iloc[test_idx].sample(4, random_state=6)
for _, row in demo_txt.iterrows():
    print('\n' + '=' * 80)
    print(f"INPUT TEXT   : \"{row['description']}\"  (TRUE: {row['category']})")
    res = assistant.assist(text=row['description'])
    print(f"PREDICTED    : {res['category']}  (conf {res['confidence']:.2f})")
    if res['warning']:
        print('WARNING      :', res['warning'])
    print('INSTRUCTIONS :')
    print('   ' + res['instructions'].replace('\n', '\n   '))
    print('SOURCES      :', [s['doc_id'] for s in res['sources']])


In [ ]:
# --------------------------------------------------------
# 5.4 Edge cases + user-feedback loop
print('--- Edge case 1: missing image file ---')
try:
    assistant.assist(image_path='definitely_missing.jpg')
except Exception as ex:
    print('Handled ->', type(ex).__name__, '-', ex)

print('\n--- Edge case 2: empty / junk text ---')
for bad in ['', '  !! ## ', 'zzz qqq']:
    try:
        r = assistant.assist(text=bad)
        print(f'  {bad!r:<12} -> {r["category"]} (conf {r["confidence"]:.2f})')
    except Exception as ex:
        print(f'  {bad!r:<12} -> Handled: {type(ex).__name__}: {ex}')

print('\n--- Edge case 3: low-confidence prediction shows a warning ---')
res = assistant.assist(text='some weird shiny rectangular thing')
print('pred:', res['category'], 'conf:', round(res['confidence'], 3),
      '| warning:', res['warning'])

print('\n--- User feedback adjusts retrieval preferences ---')
for c in ['Cardboard', 'Cardboard', 'Plastic']:
    assistant.log_feedback(c, note='resident marked useful')
boosts = assistant.apply_feedback_to_retrieval()
print('Category boosts derived from feedback log:', boosts)

# demonstrate the boost effect on a cardboard query
def _ret_top(query, k=3, boost=None):
    q = assistant.encoder.encode([query], normalize_embeddings=True)[0]
    sim = assistant.emb @ q
    A = assistant.corpus
    if boost:
        for i, r in A.iterrows():
            vc = str(r['valid_categories'])
            for c, b in boost.items():
                if c in vc:
                    sim[i] *= b
    top = np.argsort(sim)[::-1][:k]
    return [(A.iloc[i]['doc_id'], round(float(sim[i]), 3)) for i in top]

before = _ret_top('How to recycle cardboard?')
after = _ret_top('How to recycle cardboard?', boost=boosts)
print('Cardboard retrieval BEFORE feedback:', before)
print('Cardboard retrieval AFTER  feedback:', after)


### 5.5 System-level evaluation

Below we consolidate the metrics of every component into one table and record
limitations, biases and possible improvements. This mirrors the rubric's
final evaluation of the *integrated* system.


In [ ]:
# --------------------------------------------------------
# 5.5 Overall results table + reflection
summary = {
    'CNN (image)': {
        'frozen-224 head test acc': res_head[best_head]['test_acc'],
        'fine-tuned test acc': round(cnn_test_acc, 4),
        'worst class (recall)': per.sort_values('recall').iloc[0]['class'],
    },
    'Text classification': {
        'LogReg+TF-IDF test acc': round(acc, 4),
        'BiLSTM test acc': round(nn_te, 4),
        'worst error pair': pairs_worst,
    },
    'RAG retrieval': {
        'hit@3 (pure embedding)': float(ret_ev['hit@k'].mean()),
        'precision@3 (pure)': float(ret_ev['p@k'].mean()),
        'hit@3 (category-aware)': float(ret_ev_hyb['hit@k'].mean()),
        'precision@3 (category-aware)': float(ret_ev_hyb['p@k'].mean()),
    },
    'RAG generation': {
        'ROUGE-1 vs top-1 doc': round(float(ragev['R1_vs_top1'].mean()), 3),
        'ROUGE-L vs top-1 doc': round(float(ragev['ROUGE_L_vs_top1'].mean()), 3),
        'faithfulness-jaccard': round(float(ragev['jaccard_vs_ctx'].mean()), 3),
    },
}
summary_df = pd.DataFrame({k: pd.Series(v) for k, v in summary.items()})
print(summary_df.to_string())
with open(os.path.join(DATA, 'final_metrics.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

# one integrated end-to-end demo
print('\n========== END-TO-END DEMO ==========')
res = assistant.assist(text='crumpled newspaper from yesterday')
print('Q : "crumpled newspaper from yesterday"')
print('Class:', res['category'], '| conf:', round(res['confidence'], 3))
print('Instructions:')
print(' ', res['instructions'].replace('\n', '\n  '))
print('Sources:')
for s in res['sources']:
    print(f"  - [{s['score']}] {s['doc_id']}: {s['excerpt'][:90]}")


### 5.6 Reflection: limitations, biases, and next steps

**What works.** The frozen-feature MobileNetV2 head reaches ~80% test accuracy
on the 9-class RealWaste set while the 224px frozen head is the strongest single
image model; the CPU-budget 128px fine-tune lands at ~73% (the 128px downgrade
costs more than the domain adaptation gains at this resolution). TF-IDF +
LogisticRegression classifies 5-word resident descriptions near-perfectly and is
trivially explainable. The RAG component reliably *grounds*
its instructions in Metro City policy (retrieval `hit@3` near 1.0; generated
text shares vocabulary with the retrieved source).

**Limitations & biases**
1. *Image domain gap* — RealWaste photos are clean, desk-style, uniform 524px
   captures; a phone snapshot of a dirty/non-decomposed item may behave
   differently. We mitigate with augmentation but cannot fully close the gap.
2. *Glass colour roll-up* — the dataset collapses green/brown/white glass into
   one "Glass" class, hiding a colour-sorting requirement the city cares about.
3. *Class imbalance* — Plastic/Metal dominate; Textile Trash is the smallest.
   Balanced class weights mitigate accuracy bias, but the model remains less
   calibrated on rare classes (see per-class recall).
4. *Synthetic text* — `waste_descriptions.csv` is generated (templated
   adjectives), so vocabulary diversity is limited relative to genuine
   resident language; common_confusion is 50% empty.
5. *Small generator* — DistilGPT2 on CPU is heavily capacity-limited; outputs
   are correct-but-stiff and sometimes truncate. A larger instruction-tuned
   model would help readability without changing the RAG architecture.
6. *Single jurisdiction* — all policy documents are Metro City's; porting the
   system requires swapping the corpus and re-indexing.

**Suggested next steps**
- Fine-tune/calibrate on real resident text + citizen feedback logs.
- Add the colour sub-labels for glass to the CNN (hierarchical labels).
- Serve via a lightweight API (FastAPI) with the assistant behind a
  confidence gating policy so low-confidence inputs trigger human review.
- Move generation to an instruct-tuned small LM and evaluate with human raters.
